In [2]:
from dotenv import load_dotenv

load_dotenv()

True

## Sort your agent facts

Before building with a runtime context, put the facts into the right bucket:

- Context = stable, read-only metadata about the current run (for example, the user's default preferences or locale).
- State = mutable memory that changes across turns and must be persisted in the agent's checkpointed state.

If a fact is supposed to change during the conversation, it belongs in state. If it is just extra environment information, it belongs in context.

In [3]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [11]:
from langchain.agents import create_agent

agent_without_tool = create_agent(
    model="groq:openai/gpt-oss-20b",
    context_schema=ColourContext,
)

In [12]:
from langchain.messages import HumanMessage

response_without_tool = agent_without_tool.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(),
)

print(response_without_tool["messages"][-1].content)
print("This is the failure case: context exists, but there is no tool to read it.")

I’m not sure—what’s your favorite color?
This is the failure case: context exists, but there is no tool to read it.


In [ ]:
from pprint import pprint

pprint(response_without_tool)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='8c7523bf-353b-4b3a-a650-91329ce28f9b'),
              AIMessage(content='I’m not sure—I don’t have that detail. What’s your favorite colour?', additional_kwargs={'reasoning_content': 'We need to respond. The user asks "What is my favourite colour?" This is a personal question. We don\'t have that information. We can ask for clarification or guess. The user might be expecting a guess or a playful answer. According to policies, we should not fabricate personal info. We can politely say we don\'t know. Or we can ask. We can also say we don\'t know. The user might be looking for a guess. But we can\'t guess without evidence. We can ask them to tell us. We can respond: "I don\'t know. What is your favorite colour?" That is fine. Let\'s do that.'}, response_metadata={'token_usage': {'completion_tokens': 153, 'prompt_tokens': 77, 'total_tokens': 230, 'completion_time': 0.15743419

## Accessing Context

The context schema is not enough on its own. The agent needs a tool that reads the data from `runtime.context` so it can answer the question.

In [7]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [13]:
agent_with_context_tools = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext,
)

In [14]:
response_with_context = agent_with_context_tools.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(),
)

print(response_with_context["messages"][-1].content)

Your favourite colour is **blue**.


In [15]:
response_with_overridden_context = agent_with_context_tools.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green"),
)

print(response_with_overridden_context["messages"][-1].content)

Your favourite colour is **green**.
